In [1]:
%pip install -q datasets scikit-learn joblib "numpy==2.2.6" "pandas==2.2.3" "bottleneck>=1.5.0" "python-dateutil<3"

In [3]:
!pip -q install -U datasets scikit-learn pandas numpy joblib

In [4]:
from datasets import load_dataset
from collections import defaultdict
from tqdm.auto import tqdm
import pandas as pd
import numpy as np

DATASET_ID = "bvsam/cic-ids-2017"
CONFIG = "machine_learning"

MAX_PER_LABEL = 10_000
SHUFFLE_BUFFER = 100_000
SEED = 42

stream = load_dataset(
    DATASET_ID,
    CONFIG,
    split="train",
    streaming=True
)

stream = stream.shuffle(
    seed=SEED,
    buffer_size=SHUFFLE_BUFFER
)

samples = defaultdict(list)
counts = defaultdict(int)

print("Streaming records and sampling across labels...")

for row in tqdm(stream):
    label = str(row["Label"]).strip()

    if counts[label] < MAX_PER_LABEL:
        samples[label].append(row)
        counts[label] += 1

    if all(
        counts[label] >= MAX_PER_LABEL
        for label in samples
    ) and len(samples) > 0:
        # This condition alone is not sufficient to stop:
        # new labels may appear later.
        pass

    # Stop only after scanning all records, so rare
    # labels near the end of the dataset are not missed.

rows = [
    row
    for label_rows in samples.values()
    for row in label_rows
]

df = pd.DataFrame(rows)

print("\nSample shape:", df.shape)
print("\nRows per attack category:")
print(df["Label"].value_counts())

Streaming records and sampling across labels...


0it [00:00, ?it/s]


Sample shape: (79323, 79)

Rows per attack category:
Label
BENIGN                        10000
PortScan                      10000
DDoS                          10000
DoS Hulk                      10000
DoS GoldenEye                 10000
FTP-Patator                    7938
SSH-Patator                    5897
DoS slowloris                  5796
DoS Slowhttptest               5499
Bot                            1966
Web Attack � Brute Force       1507
Web Attack � XSS                652
Infiltration                     36
Web Attack � Sql Injection       21
Heartbleed                       11
Name: count, dtype: int64


In [5]:
df.columns = df.columns.str.strip()
df["Label"] = df["Label"].astype(str).str.strip()

# Remove duplicate column names
df = df.loc[:, ~df.columns.duplicated()].copy()

# Keep numeric features only
X = df.drop(columns=["Label"]).select_dtypes(
    include=["number"]
).copy()

X = X.replace([np.inf, -np.inf], np.nan)

# Drop features that are entirely missing
X = X.dropna(axis=1, how="all")

# Remove constant features
X = X.loc[:, X.nunique(dropna=False) > 1]

# Fill remaining missing values with column medians
X = X.fillna(X.median()).fillna(0)

y_multi = df.loc[X.index, "Label"].copy()

# Remove exact duplicate feature vectors.
# Conflicting labels for identical features are ambiguous;
# inspect those before treating this as a final dataset.
duplicate_mask = X.duplicated(keep="first")

print("Duplicate feature rows:", duplicate_mask.sum())

X = X.loc[~duplicate_mask].copy()
y_multi = y_multi.loc[X.index].copy()

y_binary = (y_multi != "BENIGN").astype(int)

print("\nCleaned shape:", X.shape)
print("\nMulticlass labels:")
print(y_multi.value_counts())

print("\nBinary labels:")
print(y_binary.value_counts())

Duplicate feature rows: 7840

Cleaned shape: (71483, 68)

Multiclass labels:
Label
DDoS                          10000
DoS GoldenEye                  9994
BENIGN                         9958
PortScan                       9007
DoS Hulk                       8616
FTP-Patator                    5933
DoS slowloris                  5385
DoS Slowhttptest               5228
SSH-Patator                    3219
Bot                            1953
Web Attack � Brute Force       1470
Web Attack � XSS                652
Infiltration                     36
Web Attack � Sql Injection       21
Heartbleed                       11
Name: count, dtype: int64

Binary labels:
Label
1    61525
0     9958
Name: count, dtype: int64


In [7]:
from sklearn.model_selection import train_test_split

print("Available classes:")
print(y_multi.value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_multi,
    test_size=0.20,
    random_state=42,
    stratify=y_multi
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Number of classes:", y_multi.nunique())

Available classes:
Label
DDoS                          10000
DoS GoldenEye                  9994
BENIGN                         9958
PortScan                       9007
DoS Hulk                       8616
FTP-Patator                    5933
DoS slowloris                  5385
DoS Slowhttptest               5228
SSH-Patator                    3219
Bot                            1953
Web Attack � Brute Force       1470
Web Attack � XSS                652
Infiltration                     36
Web Attack � Sql Injection       21
Heartbleed                       11
Name: count, dtype: int64

Training samples: 57186
Testing samples: 14297
Number of classes: 15


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

attack_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=24,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

attack_model.fit(X_train, y_train)

y_pred = attack_model.predict(X_test)

print("Multiclass Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)

Multiclass Accuracy: 0.9856613275512345

Classification Report:
                            precision    recall  f1-score   support

                    BENIGN       0.99      0.98      0.99      1992
                       Bot       0.96      1.00      0.98       391
                      DDoS       1.00      1.00      1.00      2000
             DoS GoldenEye       1.00      1.00      1.00      1999
                  DoS Hulk       1.00      1.00      1.00      1723
          DoS Slowhttptest       1.00      0.99      0.99      1046
             DoS slowloris       1.00      1.00      1.00      1077
               FTP-Patator       1.00      1.00      1.00      1187
                Heartbleed       1.00      1.00      1.00         2
              Infiltration       1.00      0.71      0.83         7
                  PortScan       1.00      1.00      1.00      1801
               SSH-Patator       1.00      1.00      1.00       644
  Web Attack � Brute Force       0.80      0.64    

In [9]:
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

benign_mask = y_train.astype(str).str.strip().str.upper().eq("BENIGN")
X_benign = X_train.loc[benign_mask]

if len(X_benign) < 10:
    raise ValueError(
        "Not enough benign training samples for anomaly detection."
    )

anomaly_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", IsolationForest(
        n_estimators=100,
        contamination="auto",
        random_state=42,
        n_jobs=-1
    ))
])

anomaly_model.fit(X_benign)

# 1 = normal according to Isolation Forest
# -1 = anomalous
anomaly_predictions = anomaly_model.predict(X_test)

# Larger scores mean more normal; smaller scores mean more anomalous
anomaly_scores = anomaly_model.decision_function(X_test)

print("Benign samples used:", len(X_benign))
print("Anomalies detected:", (anomaly_predictions == -1).sum())
print("Total test samples:", len(X_test))

Benign samples used: 7966
Anomalies detected: 4721
Total test samples: 14297


In [10]:
import pandas as pd

feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": attack_model.feature_importances_
}).sort_values(
    by="importance",
    ascending=False
)

display(feature_importance.head(20))

,feature,importance
0,Destination Port,0.083357
57,Init_Win_bytes_backward,0.055012
59,min_seg_size_forward,0.041263
6,Fwd Packet Length Max,0.034295
5,Total Length of Bwd Packets,0.027636
31,Fwd Header Length,0.026087
34,Bwd Packets/s,0.025124
12,Bwd Packet Length Mean,0.023807
10,Bwd Packet Length Max,0.023639
37,Packet Length Mean,0.023596


In [11]:
import numpy as np
import pandas as pd

# Recover the original rows corresponding to the cleaned feature indices.
# This preserves metadata such as IP addresses and timestamps when available.
metadata = df.loc[X.index].copy()

# Normalize column names for easier matching
metadata.columns = metadata.columns.astype(str).str.strip()

# Match X_test rows to their original records
test_metadata = metadata.loc[X_test.index].copy()

# Reset indexes for aligned alert generation
test_metadata = test_metadata.reset_index(drop=True)
X_test_rows = X_test.reset_index(drop=True)

predicted_labels = pd.Series(y_pred).reset_index(drop=True)
anomaly_flags = pd.Series(anomaly_predictions == -1)
anomaly_scores_series = pd.Series(anomaly_scores).reset_index(drop=True)


def get_value(row, possible_names, default=None):
    """Find a column using one of several possible column names."""
    lookup = {str(c).strip().lower(): c for c in row.index}

    for name in possible_names:
        col = lookup.get(name.lower())
        if col is not None:
            value = row[col]
            if pd.notna(value):
                return value

    return default


def explain_alert(features, predicted_type, is_anomaly):
    """Create a reason using measurable traffic evidence."""
    evidence = []

    def number(*names):
        value = get_value(features, names)

        try:
            value = float(value)
            if np.isfinite(value):
                return value
        except (TypeError, ValueError):
            pass

        return None

    packet_rate = number("Flow Packets/s", "Flow_Packets/s")
    syn_count = number("SYN Flag Count", "SYN_Flag_Count")
    forward_rate = number("Fwd Packets/s", "Fwd_Packets/s")
    flow_duration = number("Flow Duration", "Flow_Duration")
    destination_port = number("Destination Port", "Destination_Port")
    total_packets = number("Total Fwd Packets", "Total Fwd Packets")

    if packet_rate is not None and packet_rate > 1000:
        evidence.append(
            f"very high flow packet rate ({packet_rate:.1f} packets/s)"
        )

    if forward_rate is not None and forward_rate > 1000:
        evidence.append(
            f"very high forward packet rate ({forward_rate:.1f} packets/s)"
        )

    if syn_count is not None and syn_count > 20:
        evidence.append(
            f"elevated SYN flag count ({syn_count:.0f})"
        )

    if total_packets is not None and total_packets > 1000:
        evidence.append(
            f"large forward packet count ({total_packets:.0f})"
        )

    if flow_duration is not None and flow_duration < 1000:
        evidence.append("very short flow duration")

    if destination_port is not None:
        common_ports = {20, 21, 22, 23, 25, 53, 80, 110, 143, 443, 3389}

        if int(destination_port) not in common_ports:
            evidence.append(
                f"connection to destination port {int(destination_port)}"
            )

    attack_name = str(predicted_type).strip()

    if attack_name.upper() == "BENIGN" and is_anomaly:
        summary = "Unusual traffic pattern compared with learned benign traffic"
    elif attack_name.upper() != "BENIGN":
        summary = f"Traffic classified as {attack_name}"
    else:
        summary = "No attack category predicted"

    if evidence:
        summary += "; observed evidence: " + ", ".join(evidence)
    elif is_anomaly:
        summary += (
            "; anomaly detector found an unusual feature pattern, "
            "but no specific cause was established"
        )
    elif attack_name.upper() != "BENIGN":
        summary += (
            "; inspect the contributing traffic features to establish "
            "the specific cause"
        )

    return summary


def get_severity(attack_type, is_anomaly):
    """Prototype severity mapping; tune using measured risk."""
    name = str(attack_type).strip().lower()

    if name == "benign" and not is_anomaly:
        return "none"

    if any(word in name for word in ["ddos", "dos", "infiltration"]):
        return "high"

    if any(word in name for word in ["bot", "brute force", "web attack"]):
        return "high"

    if "portscan" in name or "port scan" in name:
        return "medium"

    if name == "benign" and is_anomaly:
        return "medium"

    return "medium"


alerts = []

for i in range(len(X_test_rows)):
    predicted_type = str(predicted_labels.iloc[i])
    is_anomaly = bool(anomaly_flags.iloc[i])

    # Create an alert if either model detects something suspicious
    if predicted_type.upper() == "BENIGN" and not is_anomaly:
        continue

    original_row = test_metadata.iloc[i]
    feature_row = X_test_rows.iloc[i]

    source = get_value(
        original_row,
        ["Source IP", "Src IP", "source", "src", "Source"]
    )

    destination = get_value(
        original_row,
        ["Destination IP", "Dst IP", "destination", "dst", "Destination"]
    )

    timestamp = get_value(
        original_row,
        ["Timestamp", "timestamp", "Flow Start"]
    )

    reason = explain_alert(
        feature_row,
        predicted_type,
        is_anomaly
    )

    alerts.append({
        "zone": "inbound" if destination is not None else "unknown",
        "type": (
            "anomaly"
            if predicted_type.upper() == "BENIGN" and is_anomaly
            else predicted_type
        ),
        "severity": get_severity(predicted_type, is_anomaly),
        "source": str(source) if source is not None else "unknown",
        "destination": (
            str(destination) if destination is not None else "unknown"
        ),
        "reason": reason,
        "timestamp": str(timestamp) if timestamp is not None else None,
        "anomaly_score": round(float(anomaly_scores_series.iloc[i]), 4),
        "sensor": "WebWatch"
    })

alerts_df = pd.DataFrame(alerts)

print("Alerts generated:", len(alerts_df))
display(alerts_df.head(20))

Alerts generated: 12482


,zone,type,severity,source,destination,reason,timestamp,anomaly_score,sensor
0,unknown,DoS GoldenEye,high,unknown,unknown,Traffic classified as DoS GoldenEye; anomaly d...,None,-0.0021,WebWatch
1,unknown,DoS Hulk,high,unknown,unknown,Traffic classified as DoS Hulk; anomaly detect...,None,-0.0156,WebWatch
2,unknown,DoS Slowhttptest,high,unknown,unknown,Traffic classified as DoS Slowhttptest; observ...,None,0.0207,WebWatch
3,unknown,DoS GoldenEye,high,unknown,unknown,Traffic classified as DoS GoldenEye; anomaly d...,None,-0.0707,WebWatch
4,unknown,DoS Slowhttptest,high,unknown,unknown,Traffic classified as DoS Slowhttptest; anomal...,None,-0.0209,WebWatch
5,unknown,DoS GoldenEye,high,unknown,unknown,Traffic classified as DoS GoldenEye; anomaly d...,None,-0.0014,WebWatch
6,unknown,anomaly,medium,unknown,unknown,Unusual traffic pattern compared with learned ...,None,-0.1037,WebWatch
7,unknown,DoS slowloris,high,unknown,unknown,Traffic classified as DoS slowloris; inspect t...,None,0.1186,WebWatch
8,unknown,PortScan,medium,unknown,unknown,Traffic classified as PortScan; observed evide...,None,0.1624,WebWatch
9,unknown,Web Attack � XSS,high,unknown,unknown,Traffic classified as Web Attack � XSS; inspec...,None,0.1351,WebWatch


In [12]:
import joblib
import os

os.makedirs("webwatch_models", exist_ok=True)

joblib.dump(
    attack_model,
    "webwatch_models/attack_classifier.pkl"
)

joblib.dump(
    anomaly_model,
    "webwatch_models/anomaly_detector.pkl"
)

joblib.dump(
    list(X.columns),
    "webwatch_models/feature_columns.pkl"
)

joblib.dump(
    list(attack_model.classes_),
    "webwatch_models/attack_classes.pkl"
)

print("Models saved successfully!")

Models saved successfully!
